In [1]:
from pathlib import Path
import sys

# This notebook is stored in <journal package>/notebooks.
notebooks_dir = Path.cwd()
if notebooks_dir.name != "notebooks":
    candidate = notebooks_dir / "notebooks"
    if candidate.exists():
        notebooks_dir = candidate

project_root = notebooks_dir.parent
nhanes_dir = project_root / "src" / "nhanes"
jordan_dir = project_root / "src" / "jordan_external_validation"
expected_nhanes = project_root / "outputs" / "nhanes"
expected_jordan = project_root / "outputs" / "jordan"

assert (project_root / "README.md").exists(), "Open this notebook from the journal package"

if str(nhanes_dir) not in sys.path:
    sys.path.insert(0, str(nhanes_dir))

print("Project root:", project_root)
print("Python executable:", sys.executable)


Project root: <project_root>
Python executable: python


In [2]:
import json
import subprocess

import pandas as pd
import matplotlib.pyplot as plt

jordan_code = jordan_dir / "external_validation"
jordan_raw = jordan_dir / "raw"
jordan_dta = jordan_raw / "jor2019.dta"
jordan_csv = jordan_raw / "jor2019.csv"
jordan_output = jordan_code / "jordan_results"

print("Jordan code folder:", jordan_code)
print("Licensed DTA present:", jordan_dta.exists())
print("Licensed CSV present:", jordan_csv.exists())


Jordan code folder: <project_root>\src\jordan_external_validation\external_validation
Licensed DTA present: False
Licensed CSV present: False


In [3]:
freeze_lock = jordan_code / "transport_model_freeze.lock.json"
freeze_manifest = jordan_code / "frozen_nhanes_models" / "freeze_manifest.json"
crosswalk_file = jordan_code / "variable_crosswalk.csv"
harmonization_file = jordan_code / "HARMONIZATION_PROTOCOL.md"

assert freeze_lock.exists()
assert freeze_manifest.exists()
assert crosswalk_file.exists()
assert harmonization_file.exists()

freeze_information = json.loads(freeze_lock.read_text(encoding="utf-8"))
model_manifest = json.loads(freeze_manifest.read_text(encoding="utf-8"))
variable_crosswalk = pd.read_csv(crosswalk_file)

display(freeze_information)
display(model_manifest)
display(variable_crosswalk)


{'frozen_at_utc': '2026-10-20T02:10:52.920878+00:00',
 'python': '3.11.6',
 'seed': 42,
 'threshold': 0.2,
 'model_family': 'logistic regression only',
 'calibration': 'weighted logistic calibration fitted in NHANES validation partition',
 'external_refitting_permitted': False,
 'external_recalibration_permitted': False,
 'participant_level_jordan_data_accessed_before_freeze': False,
 'synthetic_data_used': False,
 'protocol_sha256': '6c0e308124b33317524ba946a5fe7748e25b356bc318496731d92a04d5c3e08c',
 'crosswalk_sha256': '1dd2ccb72b950097525675664f7739174a77e9fcb55df4aa4dd605f46d895847',
 'nhanes_psu_assignment_sha256': '3497e1ed494ff6b1c85ea350dc5fe0d4cef904850830106704ebcfdc2242e733',
 'objects': {'BASE_LogReg_frozen.joblib': '26df478c373fd3ba4b4b65417f6cb5f93b4f262dce1e28bc682cdedd51389e3e',
  'BMI_LogReg_frozen.joblib': '8d5906487bc28eda9cecbd5dc3da0b55940888ac9d0615c218d1e6bd67ccd682',
  'WHR_LogReg_frozen.joblib': 'dfb1bf08070e205d2ff17b05fc4772270ca1f38f04dfb0f93642d48ef85f18a0'

{'frozen_at_utc': '2026-10-20T02:10:52.920878+00:00',
 'python': '3.11.6',
 'seed': 42,
 'threshold': 0.2,
 'model_family': 'logistic regression only',
 'calibration': 'weighted logistic calibration fitted in NHANES validation partition',
 'external_refitting_permitted': False,
 'external_recalibration_permitted': False,
 'participant_level_jordan_data_accessed_before_freeze': False,
 'synthetic_data_used': False,
 'protocol_sha256': '6c0e308124b33317524ba946a5fe7748e25b356bc318496731d92a04d5c3e08c',
 'crosswalk_sha256': '1dd2ccb72b950097525675664f7739174a77e9fcb55df4aa4dd605f46d895847',
 'nhanes_psu_assignment_sha256': '3497e1ed494ff6b1c85ea350dc5fe0d4cef904850830106704ebcfdc2242e733',
 'objects': {'BASE_LogReg_frozen.joblib': '26df478c373fd3ba4b4b65417f6cb5f93b4f262dce1e28bc682cdedd51389e3e',
  'BMI_LogReg_frozen.joblib': '8d5906487bc28eda9cecbd5dc3da0b55940888ac9d0615c218d1e6bd67ccd682',
  'WHR_LogReg_frozen.joblib': 'dfb1bf08070e205d2ff17b05fc4772270ca1f38f04dfb0f93642d48ef85f18a0'

,analysis_variable,nhanes_source,jordan_source,jordan_unit_or_codes,external_model_status,reason_or_transformation
0,age,RIDAGEYR,age,years,include,Restrict both cohorts to men aged 18-69
1,bmi,BMXBMI,m11 and m12,cm and kg,include,Derive m12/(m11/100)^2
2,waist_to_hip_ratio,BMXWAIST and BMXHIP,m14 and m15,cm,include,Derive m14/m15
3,systolic_bp,BPXSY1-BPXSY4,m4a m5a m6a,mmHg,include,Mean valid readings requiring at least two
4,diastolic_bp,BPXDI1-BPXDI4,m4b m5b m6b,mmHg,include,Mean valid readings requiring at least two
5,heart_rate,BPXPLS,m16a m16b m16c,beats/min,include,Mean valid readings requiring at least two
6,hypertension_history,BPQ020,h2a,1=yes; 2=no,include,Missing remains missing and is handled by froz...
7,education_level,DMDEDUC2,c4 or c5,country-specific,exclude,No exact category harmonization
8,ethnicity,RIDRETH3,none,not available,exclude,No harmonized construct
9,poverty_income_ratio,INDFMPIR,c10/c11,country-specific,exclude,No harmonized construct


In [4]:
frozen_model_dir = jordan_code / "frozen_nhanes_models"
frozen_objects = sorted(frozen_model_dir.glob("*_LogReg_frozen.joblib"))

for path in frozen_objects:
    print(path.name)

assert len(frozen_objects) == 4
print("PASS: four frozen logistic model objects are present.")


BASE_LogReg_frozen.joblib
BMI_LogReg_frozen.joblib
BOTH_LogReg_frozen.joblib
WHR_LogReg_frozen.joblib
PASS: four frozen logistic model objects are present.


In [5]:
# Run automatically for an authorized local user when both licensed files are
# present. In the clean journal ZIP the files are absent, so packaged aggregate
# results are used safely instead.
RUN_JORDAN_EXTERNAL_VALIDATION = jordan_dta.exists() and jordan_csv.exists()
print("Run participant-level external validation:", RUN_JORDAN_EXTERNAL_VALIDATION)

run_command = [
    sys.executable,
    "-u",
    str(jordan_code / "run_jordan_external_validation.py"),
    str(jordan_dta),
    "--output",
    str(jordan_output),
    "--replicates",
    "2000",
]

print("Command:", " ".join(run_command))

if RUN_JORDAN_EXTERNAL_VALIDATION:
    assert jordan_dta.exists(), "Licensed jor2019.dta is missing"
    assert jordan_csv.exists(), "Licensed jor2019.csv is missing"

    subprocess.run(run_command, cwd=jordan_dir, check=True)
    subprocess.run(
        [sys.executable, str(jordan_code / "verify_jordan_external_validation.py")],
        cwd=jordan_dir,
        check=True,
    )
    subprocess.run(
        [sys.executable, str(jordan_code / "create_jordan_results_summary.py")],
        cwd=jordan_dir,
        check=True,
    )
    print("PASS: external validation and independent verification completed.")
else:
    print("Authorized rerun skipped. Packaged aggregate results will be inspected.")


Run participant-level external validation: False
Command: python -u <project_root>\src\jordan_external_validation\external_validation\run_jordan_external_validation.py <project_root>\src\jordan_external_validation\raw\jor2019.dta --output <project_root>\src\jordan_external_validation\external_validation\jordan_results --replicates 2000
Authorized rerun skipped. Packaged aggregate results will be inspected.


In [6]:
jordan_results = (
    jordan_output
    if (jordan_output / "jordan_external_validation_headline_results.csv").exists()
    else expected_jordan
)

print("Reading Jordan results from:", jordan_results)


Reading Jordan results from: <project_root>\outputs\jordan


In [7]:
print("Results folder:", jordan_results.resolve())
print(
    "Fresh results available:",
    (jordan_output / "jordan_external_validation_headline_results.csv").exists()
)
print("Using fresh participant-level results:", jordan_results == jordan_output)

Results folder: <project_root>\outputs\jordan
Fresh results available: False
Using fresh participant-level results: False


In [8]:
cohort_flow = pd.read_csv(jordan_results / "jordan_cohort_flow.csv")
missingness = pd.read_csv(jordan_results / "jordan_predictor_missingness.csv")

display(cohort_flow)
display(missingness)

manifest = json.loads(
    (jordan_results / "jordan_external_validation_manifest.json").read_text(encoding="utf-8")
)
display(manifest)


,stage,n
0,Public-use file,5713
1,Men aged 18-69,2203
2,Men aged 18-69 with positive wstep2,2202
3,Previously had blood sugar measured (h6=1),885
4,Primary external outcome observed,885


,variable,missing_n,missing_percent
0,age,0,0.000000
1,systolic_bp,79,8.926554
2,diastolic_bp,79,8.926554
3,heart_rate,79,8.926554
4,bmi,89,10.056497
5,waist_to_hip_ratio,99,11.186441
6,hypertension_history,59,6.666667


{'source_file': 'jor2019.dta',
 'source_sha256': 'c5e32adda1af8300a198b52a1e82ea003a6144008637b88d7623eaba0e5b93d9',
 'frozen_lock_sha256': 'f20184a605d0f4b40f0aabf7e59b31850665ef79ad27c618de0b026e0fca870b',
 'frozen_at_utc': '2026-10-20T02:10:52.920878+00:00',
 'model_objects_verified': True,
 'model_refit': False,
 'recalibration': False,
 'threshold_optimization': False,
 'synthetic_data_used': False,
 'outcome': 'h7a=1 versus h7a=2 among h6=1; related but nonidentical to NHANES DIQ010',
 'analysis_weight': 'wstep2',
 'resampling_replicates_requested': 2000,
 'resampling_replicates_valid': 2000,
 'n_strata': 32,
 'n_psus': 507,
 'resampling_design_universe': 'all public-file participants with positive wstep2; metrics restricted to the eligible h6=1 outcome domain',
 'resampling_universe_n_strata': 32,
 'resampling_universe_n_psus': 750,
 'nationality_sensitivity': [{'nationality': 'Jordanian',
   'status': 'completed',
   'n': 564,
   'cases': 161,
   'noncases': 403,
   'n_strata':

In [9]:
headline = pd.read_csv(
    jordan_results / "jordan_external_validation_headline_results.csv"
)
full_results = pd.read_csv(
    jordan_results / "jordan_external_validation_results.csv"
)

display(headline.round(4))
display(full_results.round(4))


,configuration,n,cases,weighted_auc,auc_lower_psu,auc_upper_psu,brier,calibration_slope,calibration_in_the_large,sensitivity_at_0_20,specificity_at_0_20
0,BASE,885,249,0.7694,0.7159,0.8174,0.1725,0.6541,1.3032,0.3589,0.8976
1,BMI,885,249,0.7617,0.7096,0.8109,0.1797,0.6141,1.5387,0.3024,0.9204
2,WHR,885,249,0.7378,0.6858,0.7872,0.1888,0.4248,1.8005,0.2562,0.9089
3,BOTH,885,249,0.7391,0.6876,0.7880,0.1898,0.4253,1.8728,0.2593,0.9213


,config,n,cases,threshold,pr_auc,auc,brier,average_precision,sensitivity,specificity,ppv,npv,accuracy,f1,calibration_intercept_joint,calibration_slope,calibration_fit_converged,calibration_in_the_large
0,BASE,885,249,0.2,0.4496,0.7694,0.1725,0.4554,0.3589,0.8976,0.5115,0.8241,0.7737,0.4218,0.5364,0.6541,True,1.3032
1,BMI,885,249,0.2,0.4435,0.7617,0.1797,0.4497,0.3024,0.9204,0.5316,0.8153,0.7782,0.3855,0.5905,0.6141,True,1.5387
2,WHR,885,249,0.2,0.4076,0.7378,0.1888,0.4141,0.2562,0.9089,0.4565,0.8035,0.7587,0.3282,0.2300,0.4248,True,1.8005
3,BOTH,885,249,0.2,0.4099,0.7391,0.1898,0.4157,0.2593,0.9213,0.4961,0.8063,0.7690,0.3406,0.2675,0.4253,True,1.8728


In [10]:
contrasts = pd.read_csv(
    jordan_results / "jordan_external_validation_auc_contrasts.csv"
)
psu_intervals = pd.read_csv(
    jordan_results / "jordan_external_validation_psu_intervals.csv"
)

display(contrasts.round(4))
display(psu_intervals.round(4))


,contrast,auc_difference,lower_psu,upper_psu
0,BMI minus WHR,0.0239,-0.0018,0.0559
1,WHR minus BMI,-0.0239,-0.0559,0.0018
2,BMI plus WHR minus BMI,-0.0226,-0.0529,0.0021


,metric,lower_psu,upper_psu
0,BASE_pr_auc,0.3762,0.5350
1,BASE_auc,0.7159,0.8174
2,BASE_brier,0.1429,0.2038
3,BASE_average_precision,0.3850,0.5464
4,BASE_sensitivity,0.2817,0.4386
5,BASE_specificity,0.8663,0.9253
6,BASE_ppv,0.4088,0.6118
7,BASE_npv,0.7849,0.8624
8,BASE_accuracy,0.7370,0.8110
9,BASE_f1,0.3412,0.4986


In [11]:
complete_case = pd.read_csv(
    jordan_results / "jordan_complete_case_sensitivity.csv"
)
nationality = pd.read_csv(
    jordan_results / "jordan_nationality_sensitivity.csv"
)
jordanian_intervals = pd.read_csv(
    jordan_results / "jordan_jordanian_psu_intervals.csv"
)
syrian_intervals = pd.read_csv(
    jordan_results / "jordan_syrian_psu_intervals.csv"
)

display(complete_case.round(4))
display(nationality.round(4))
display(jordanian_intervals.round(4))
display(syrian_intervals.round(4))


,config,n,cases,pr_auc,auc,brier,average_precision,sensitivity,specificity,ppv,npv,accuracy,f1,calibration_intercept_joint,calibration_slope,calibration_fit_converged,calibration_in_the_large
0,BASE,734,207,0.4380,0.7609,0.1726,0.4449,0.3658,0.8854,0.4891,0.8231,0.7654,0.4186,0.4334,0.6368,True,1.2063
1,BMI,734,207,0.4349,0.7567,0.1791,0.4423,0.3082,0.9091,0.5043,0.8141,0.7704,0.3826,0.4869,0.5989,True,1.4308
2,WHR,734,207,0.4017,0.7288,0.1890,0.4094,0.2702,0.8996,0.4469,0.8042,0.7544,0.3368,0.1285,0.4006,True,1.7199
3,BOTH,734,207,0.4031,0.7286,0.1901,0.4099,0.2715,0.9081,0.4698,0.8060,0.7611,0.3441,0.1531,0.3977,True,1.7944


,nationality,config,n,cases,threshold,pr_auc,auc,brier,average_precision,sensitivity,specificity,ppv,npv,accuracy,f1,calibration_intercept_joint,calibration_slope,calibration_fit_converged,calibration_in_the_large
0,Jordanian,BASE,564,161,0.2,0.4433,0.7660,0.1740,0.4499,0.3605,0.8948,0.5077,0.8230,0.7713,0.4216,0.5035,0.6414,True,1.2928
1,Jordanian,BMI,564,161,0.2,0.4369,0.7574,0.1812,0.4441,0.3030,0.9170,0.5235,0.8139,0.7750,0.3838,0.5500,0.5999,True,1.5264
2,Jordanian,WHR,564,161,0.2,0.4037,0.7352,0.1901,0.4110,0.2566,0.9048,0.4478,0.8018,0.7549,0.3262,0.2024,0.4164,True,1.7803
3,Jordanian,BOTH,564,161,0.2,0.4060,0.7363,0.1910,0.4125,0.2599,0.9172,0.4857,0.8047,0.7652,0.3386,0.2379,0.4166,True,1.8510
4,Syrian,BASE,321,88,0.2,0.5589,0.8193,0.1518,0.5696,0.3350,0.9360,0.5857,0.8389,0.8081,0.4262,1.1732,0.8816,True,1.4673
5,Syrian,BMI,321,88,0.2,0.5736,0.8216,0.1586,0.5783,0.2928,0.9678,0.7105,0.8351,0.8242,0.4147,1.4061,0.8799,True,1.7349
6,Syrian,WHR,321,88,0.2,0.5003,0.7709,0.1709,0.5074,0.2496,0.9658,0.6637,0.8265,0.8134,0.3627,0.8850,0.6107,True,2.0986
7,Syrian,BOTH,321,88,0.2,0.5046,0.7748,0.1721,0.5116,0.2496,0.9785,0.7582,0.8283,0.8234,0.3755,0.9614,0.6148,True,2.1976


,metric,lower_psu,upper_psu
0,BASE_pr_auc,0.3698,0.5287
1,BASE_auc,0.7122,0.8168
2,BASE_brier,0.1425,0.2090
3,BASE_average_precision,0.3792,0.5398
4,BASE_sensitivity,0.2764,0.4482
5,BASE_specificity,0.8620,0.9234
6,BASE_ppv,0.4064,0.6065
7,BASE_npv,0.7763,0.8636
8,BASE_accuracy,0.7283,0.8100
9,BASE_f1,0.3374,0.4994


,metric,lower_psu,upper_psu
0,BASE_pr_auc,0.4324,0.6807
1,BASE_auc,0.7474,0.8812
2,BASE_brier,0.1172,0.1902
3,BASE_average_precision,0.4499,0.6935
4,BASE_sensitivity,0.2348,0.4474
5,BASE_specificity,0.8976,0.9644
6,BASE_ppv,0.4148,0.7422
7,BASE_npv,0.7896,0.8840
8,BASE_accuracy,0.7569,0.8532
9,BASE_f1,0.3084,0.5379


In [12]:
figure_file = jordan_results / "jordan_external_validation_auc.png"

if figure_file.exists():
    figure = plt.imread(figure_file)
    plt.figure(figsize=(9, 5))
    plt.imshow(figure)
    plt.axis("off")
    plt.show()
else:
    print("Figure file is created by an authorized external-validation rerun.")


Figure file is created by an authorized external-validation rerun.


In [13]:
external_verification = json.loads(
    (jordan_results / "jordan_external_validation_verification.json").read_text(encoding="utf-8")
)
integrated_verification = json.loads(
    (
        jordan_results / "integrated_document_verification.json"
        if (jordan_results / "integrated_document_verification.json").exists()
        else expected_jordan / "integrated_document_verification.json"
    ).read_text(encoding="utf-8")
)

display(external_verification)
display(integrated_verification)

assert str(external_verification["status"]).lower() == "pass"
assert integrated_verification["status"] == "PASS"
print("PASS: external pipeline and document integration are verified.")


{'status': 'pass',
 'verified_at_local_date': '2026-10-27',
 'checks': [{'check': 'frozen_object_hashes', 'status': 'pass', 'count': 4},
  {'check': 'source_and_lock_manifest', 'status': 'pass'},
  {'check': 'cohort_and_design',
   'status': 'pass',
   'eligible_n': 885,
   'cases': 249,
   'domain_strata': 32,
   'domain_psus': 507,
   'universe_psus': 750},
  {'check': 'csv_dta_harmonization', 'status': 'pass'},
  {'check': 'independent_metric_recalculation',
   'status': 'pass',
   'values': 52},
  {'check': 'resampling_endpoints',
   'status': 'pass',
   'replicate_sets': 3,
   'replicates_each': 2000,
   'interval_pairs': 165}]}

{'status': 'PASS',
 'checks': {'manuscript_zip_valid': True,
  'supplement_zip_valid': True,
  'manuscript_counts': True,
  'supplement_counts': True,
  'original_manuscript_tables_unchanged': True,
  'original_supplement_tables_unchanged': True,
  'forbidden_absent::No external validation has been performed': True,
  'forbidden_absent::real external validation remains unavailable': True,
  'forbidden_absent::No suitable real independent cohort': True,
  'forbidden_absent::future T2DM risk prediction': True,
  'forbidden_absent::early prediction': True,
  'forbidden_absent::high-risk individuals': True,
  'manuscript_required::External validation in Jordan STEPS 2019': True,
  'manuscript_required::0.7617': True,
  'manuscript_required::0.7378': True,
  'manuscript_required::0.0239': True,
  'manuscript_required::-0.0018 to 0.0559': True,
  'manuscript_required::-0.0226': True,
  'manuscript_required::-0.0529 to 0.0021': True,
  'manuscript_required::licensed Jordan participant-level d

PASS: external pipeline and document integration are verified.


In [14]:
bmi = headline.query("configuration == 'BMI'").iloc[0]
whr = headline.query("configuration == 'WHR'").iloc[0]
both = headline.query("configuration == 'BOTH'").iloc[0]

assert round(bmi["weighted_auc"], 4) == 0.7617
assert round(whr["weighted_auc"], 4) == 0.7378
assert round(both["weighted_auc"], 4) == 0.7391

bmi_minus_whr = contrasts.query("contrast == 'BMI minus WHR'").iloc[0]
both_minus_bmi = contrasts.query("contrast == 'BMI plus WHR minus BMI'").iloc[0]

assert round(bmi_minus_whr["auc_difference"], 4) == 0.0239
assert round(bmi_minus_whr["lower_psu"], 4) == -0.0018
assert round(bmi_minus_whr["upper_psu"], 4) == 0.0559

assert round(both_minus_bmi["auc_difference"], 4) == -0.0226
assert round(both_minus_bmi["lower_psu"], 4) == -0.0529
assert round(both_minus_bmi["upper_psu"], 4) == 0.0021

print("PASS: principal Jordan external-validation values match the manuscript.")


PASS: principal Jordan external-validation values match the manuscript.
